# TP PAC-Bayes -- Session 4: PAC-Bayesian learning of linear classifiers

Solution notebook for Session 4 (three hours, Parts 4.1 to 4.4 of the sheet `tp_pacbayes.pdf`). It reproduces the numbers, tables (written to `results/`) and figures (written to `figures/`) of the solution sheet `tp_pacbayes_solutions.pdf`.

It needs `pbtools.py`, `prepdata.py` and the folder `datasets/` of the course, either in this folder or in its parent folder. Each session notebook is self-contained: the helper functions it needs are redefined at the top.

In [1]:
import os, sys, time
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler

HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
for p in [HERE, os.path.join(HERE, ".."), os.path.join(HERE, "..", "data"), os.path.join(HERE, "..", "..", "common"), os.path.join(HERE, "data"), os.path.join(HERE, "..", "common")]:
    if p not in sys.path:
        sys.path.insert(0, p)
from pbtools import (kl_bin, kl_inv_upper, kl_inv_lower, kl_div, bound_seeger, bound_mcallester,
                     BaggedTrees, mv_statistics, fo_bound, tnd_bound, cbound_pb, softmax,
                     optimize_fo, optimize_tnd)
from prepdata import data_recovery

DATADIR = next((d for d in [os.path.join(HERE, "data"), HERE, os.path.join(HERE, ".."), os.path.join(HERE, "..", "data")] if os.path.isdir(os.path.join(d, "datasets"))), HERE)
RES = os.path.join(HERE, "results"); FIG = os.path.join(HERE, "figures")
os.makedirs(RES, exist_ok=True); os.makedirs(FIG, exist_ok=True)
DELTA = 0.05
N_SPLITS = 5
BALANCED = ["australian", "balance", "bupa", "german", "heart", "iono", "pima", "sonar", "wdbc", "spambase"]
IMBALANCED = ["abalone8", "pageblocks", "satimage", "segmentation", "yeast3"]
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False})


def load(name):
    """Load a dataset of the course with labels in {-1,+1} and standardized features."""
    cwd = os.getcwd(); os.chdir(DATADIR)
    try:
        X, y = data_recovery(name)
    finally:
        os.chdir(cwd)
    X = StandardScaler().fit_transform(np.asarray(X, dtype=float))
    return X, np.where(np.asarray(y) > 0, 1, -1)


def eff(rho):
    """Effective number of voters 1/sum rho^2."""
    return 1.0 / np.sum(rho ** 2)


def save_tab(name, rows):
    open(os.path.join(RES, name), "w").write("\n".join(rows) + "\n")
    print("  -> results/" + name)


def ms(a, d=3):
    a = np.asarray(a, dtype=float)
    return f"{a.mean():.{d}f} ({a.std():.{d}f})"

## Part 4.1 -- PBGD and Monte Carlo certificates

In [2]:
from scipy.stats import norm as _norm
from scipy.optimize import minimize as _minimize
from sklearn.svm import LinearSVC


def add_bias(X):
    return np.hstack([X, np.ones((X.shape[0], 1))])


def probit_risk(w, X, y):
    """Exact empirical Gibbs risk of the Gaussian posterior N(w, I) on linear classifiers."""
    return np.mean(_norm.cdf(-y * (X @ w) / np.linalg.norm(X, axis=1)))


def pbgd(X, y, C, w0=None, w_init=None):
    """Minimize C * sum_i Phi(-y_i <w,x_i>/||x_i||) + ||w - w0||^2 / 2 (prior N(w0, I))."""
    nx = np.linalg.norm(X, axis=1); w0 = np.zeros(X.shape[1]) if w0 is None else w0
    def obj(w):
        t = -y * (X @ w) / nx
        return C * np.sum(_norm.cdf(t)) + 0.5 * (w - w0) @ (w - w0), C * X.T @ (_norm.pdf(t) * (-y / nx)) + (w - w0)
    start = w0 if w_init is None else w_init
    return _minimize(obj, start, jac=True, method="L-BFGS-B").x


def mc_certificate(w, w0, X, y, N, delta, rng):
    """Certificate estimated with N sampled classifiers (delta/2 for the Monte Carlo step, delta/2 for Seeger)."""
    V = w + rng.randn(N, len(w))
    emp = np.mean(np.sign(X @ V.T) != y[:, None])
    up = kl_inv_upper(emp, np.log(2 / (delta / 2)) / N)
    return bound_seeger(up, (w - w0) @ (w - w0) / 2, len(y), delta / 2)


def partD():
    print("Part D (PBGD)")
    rows = []; Cs = np.logspace(-2, 2, 13); curves = {}
    for name in ["australian", "pima", "wdbc", "spambase"]:
        X, y = load(name); X = add_bias(X); R = []
        for seed in range(3):
            Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=seed)
            svm = LinearSVC(C=1.0, fit_intercept=False, max_iter=20000).fit(Xtr, ytr)
            w_svm = svm.coef_.ravel(); best = None; curve = []
            for C in Cs:
                w = pbgd(Xtr, ytr, C, w_init=w_svm)
                g = probit_risk(w, Xtr, ytr); b = bound_seeger(g, w @ w / 2, len(ytr), DELTA)
                curve.append(b)
                if best is None or b < best[0]:
                    best = (b, C, w, g)
            if seed == 0:
                curves[name] = curve
            b, C, w, g = best
            mc = mc_certificate(w, np.zeros_like(w), Xtr, ytr, 2000, DELTA, np.random.RandomState(seed))
            R.append((np.mean(svm.predict(Xte) != yte), C, np.linalg.norm(w), g, b, mc,
                      np.mean(np.sign(Xte @ w) != yte), probit_risk(w, Xte, yte)))
        R = np.mean(R, 0)
        rows.append(f"{name} & {R[0]:.3f} & {R[1]:.2f} & {R[2]:.2f} & {R[3]:.3f} & {R[4]:.3f} & {R[5]:.3f} & {R[6]:.3f} & {R[7]:.3f} \\\\")
    save_tab("pD_pbgd.tex", rows)
    fig, ax = plt.subplots(figsize=(5.5, 2.8))
    for name, c in curves.items():
        ax.plot(Cs, c, marker="o", ms=3, label=name)
    ax.set_xscale("log"); ax.set_xlabel("$C$"); ax.set_ylabel("Seeger certificate"); ax.set_ylim(0, 1); ax.legend(fontsize=7)
    fig.tight_layout(); fig.savefig(os.path.join(FIG, "tp_pbgd.pdf")); plt.close(fig)

In [3]:
t0 = time.time(); partD(); print(f'{time.time() - t0:.0f} s')

Part D (PBGD)


  -> results/pD_pbgd.tex


42 s


## Part 4.2 -- Data-dependent Gaussian prior

In [4]:
def partE():
    print("Part E (data-dependent prior)")
    rows = []; Cs = np.logspace(-2, 2, 13)
    for name in ["australian", "pima", "wdbc", "spambase"]:
        X, y = load(name); X = add_bias(X)
        for frac in [0.0, 0.2, 0.5]:
            R = []
            for seed in range(3):
                Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=seed)
                if frac > 0:
                    X1, X2, y1, y2 = train_test_split(Xtr, ytr, train_size=frac, stratify=ytr, random_state=seed)
                    w0 = pbgd(X1, y1, 1.0)   # prior centre learned on S1 (PBGD with prior N(0,I))
                else:
                    X2, y2, w0 = Xtr, ytr, np.zeros(X.shape[1])
                best = None
                for C in Cs:
                    w = pbgd(X2, y2, C, w0=w0, w_init=w0)
                    b = bound_seeger(probit_risk(w, X2, y2), (w - w0) @ (w - w0) / 2, len(y2), DELTA)
                    if best is None or b < best[0]:
                        best = (b, w)
                b, w = best
                R.append((len(y2), (w - w0) @ (w - w0) / 2, b, np.mean(np.sign(Xte @ w) != yte)))
            R = np.mean(R, 0)
            rows.append(f"{name} & {frac:.1f} & {int(R[0])} & {R[1]:.2f} & {R[2]:.3f} & {R[3]:.3f} \\\\")
    save_tab("pE_ddprior.tex", rows)

In [5]:
t0 = time.time(); partE(); print(f'{time.time() - t0:.0f} s')

Part E (data-dependent prior)


  -> results/pE_ddprior.tex
111 s


## Parts 4.3 and 4.4 -- Synthesis and report

Part 4.3 is a written synthesis (decision table comparing all the certificates met in the four sessions) and Part 4.4 is the report; see the solution sheet for a model answer. No code is needed.